# Tarefa 5 — Árvore Final, Abertura Única do Teste e Validação em Fluxos Não Vistos

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Equipe:** Grupo 42 — Wendell Rodrigues Macedo (Scrum Master), Vinicius Firmino Souto, Fernando Paiva Coelho, João Gustavo Sanches, Daniel Dias Paranhos

Este notebook executa e audita todos os entregáveis da **Tarefa 5**:
1. Verificação da árvore congelada na Tarefa 4 (`criterion="entropy"`, `max_depth=6`, `min_samples_leaf=20`, `random_state=42`, `7` folhas) e das 7 regras em português;
2. **Abertura única do Teste Temporal** (`data/processed/teste_periodo_b.csv` — `48.270` medições);
3. **Protocolo B (Fluxo que a árvore não viu):** hold-out de `16` fluxos inteiros das sondas `*03` (`40.201` medições) com baseline próprio calculado exclusivamente no Período A (`>= 1.500` RTTs válidos), além do registro dos `3` fluxos excluídos por `baseline_insuficiente`;
4. **Declaração formal sobre o horizonte de 12 minutos:** comparação contra a regra de persistência (`repetir a classe atual`), demonstrando por que o entregável é formalmente um **Detector do estado atual** relativo ao baseline.

In [1]:
import json
from pathlib import Path
from src.arvore_final import executar_pipeline_tarefa5

dir_proj = Path('..').resolve()
with open(dir_proj / 'metadados' / 'resumo_tarefa5.json', encoding='utf-8') as f:
    r5 = json.load(f)
print(r5['secao1_arvore_medida']['diagrama_texto'])

=== 1. CONTRATO E ÁRVORE FINAL CONGELADA (TAREFA 4 -> TAREFA 5) ===
Treino: 80386 | Validação: 32148 | Teste Único (aberto 1x): 48270
Profundidade escolhida olhando o teste? False
Baseline do teste usou medição do teste? False
Colunas usadas (X): ['z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco']

Diagrama da Árvore Final (criterion=entropy, max_depth=6, min_samples_leaf=20, 7 folhas):
|--- z_robusto <= 3.5010 (n=80386, entropy=1.0058)
|   |--- perda_pct <= 16.6650 (n=60817, entropy=0.8028)
|   |   |--- n5_aumento80 <= 1.5000 (n=50520, entropy=0.3902)
|   |   |   |--- n5_timeout <= 2.5000 (n=48629, entropy=0.2319)
|   |   |   |   |--- z_robusto <= 2.0003 (n=47257, entropy=0.0479)
|   |   |   |   |   |--- classe: OK (n=46212, [OK=46212, RISCO=0, FALHA=0], entropy=-0.0000)
|   |   |   |   |--- z_robusto >  2.0003
|   |   |   |   |   |--- n5_risco <= 1.5000 (n=1045, entropy=0.7969)
|   |   |   |   |   |   |--- classe: OK

In [2]:
# Avaliação única no conjunto de teste temporal (teste_periodo_b.csv — 48.270 linhas)
av_te = r5['secao2_teste_temporal_unico']['avaliacao_arvore_final_teste']
print(json.dumps(av_te, indent=2, ensure_ascii=False))

=== 2. TESTE TEMPORAL ÚNICO (teste_periodo_b.csv — 48.270 linhas) ===
F1 Macro no Teste: 1.0000
Acurácia no Teste: 1.0000 (Total de erros = 0)
Matriz de Confusão 3x3 (Teste Único — Contagem):
  Real OK    -> OK=28303, RISCO=0, FALHA=0
  Real RISCO -> OK=0, RISCO=189, FALHA=0
  Real FALHA -> OK=0, RISCO=0, FALHA=19778

Métricas por Classe no Teste:
  OK    -> Precisão=1.0000, Recall=1.0000, F1=1.0000 (suporte=28303)
  RISCO -> Precisão=1.0000, Recall=1.0000, F1=1.0000 (suporte=189)
  FALHA -> Precisão=1.0000, Recall=1.0000, F1=1.0000 (suporte=19778)

Casos Auditados no Teste:
  1) Caminho longo estável (OK): 301|14.139.0.1 (BR->IN) em 2026-09-18T02:32:00Z | RTT=342.77 ms (mediana=339.2500 ms, z=0.9347) -> Real=OK, Predito=OK
  2) Fronteira RISCO<->FALHA: 301|155.232.0.1 (BR->ZA) em 2026-09-20T03:56:00Z | RTT=358.20 ms (z=3.4160, n5_risco=2) -> Real=RISCO, Tarefa 3=FALHA, Árvore Final=RISCO
  3) Caminho curto com timeout persistente: 101|194.109.133.10 (DE->DE) em 2026-09-18T04:20:00Z | 

In [3]:
# Protocolo B (Fluxos não vistos no treino) e Comparação no Horizonte de 12 Minutos
print(json.dumps(r5['secao3_fluxo_nao_visto']['protocolo_b_holdout_fluxos_inteiros']['avaliacao_global_fluxos_novos'], indent=2, ensure_ascii=False))
print(json.dumps(r5['secao4_declaracoes_e_horizonte_12min']['comparacao_horizonte_12min_teste'], indent=2, ensure_ascii=False))

=== 3. CHECAGEM DE FLUXOS NÃO VISTOS NO TREINO (PROTOCOLO B) E HORIZONTE DE 12 MIN ===
Fluxos excluídos por baseline insuficiente (< 1.500 RTTs válidos no Período A): ['301|170.81.100.114', '302|170.81.100.114', '303|170.81.100.114']
Protocolo B (Hold-out de 16 fluxos inteiros das sondas *03, N=40201 medições): F1 Macro = 1.0000, Erros = 0
  - Exemplo longo estável não visto: 303|203.178.136.1 (BR->JP) em 2026-09-13T04:32:00Z | RTT=277.91 ms -> Predito=OK
  - Exemplo curto degradado não visto: 103|194.109.133.10 (DE->DE) em 2026-09-13T04:48:00Z | RTT=18.31 ms (z=5.1069) -> Predito=FALHA

Comparação no Horizonte de 12 Minutos à Frente (N=48078 pares no teste):
  - Regra Persistência (repetir classe atual): F1 Macro = 0.4610 | Recall FALHA = 0.6311 | Recall RISCO = 0.0106
  - Árvore Preditora 12 min:                   F1 Macro = 0.4816 | Recall FALHA = 0.5329 | Recall RISCO = 0.0000
  -> Conclusão: No horizonte de 12 minutos à frente (3 ciclos de 240s), a árvore (F1 macro = 0.4816, Recal